In [ ]:
# run this notebook after 5_subset_vds_python
# use this notebook to call differences between twins
# after this, run 7_analyze_differences_twins_R

In [ ]:
import hail as hl
import os
import pandas
from google.cloud import storage
import re
import numpy as np 

In [ ]:
bucket=os.getenv('WORKSPACE_BUCKET')
hl.init(idempotent=True)
hl.default_reference('GRCh38')
f_twins = bucket + "/relatedness/twin_dup.txt"
twins_all = pandas.read_table(f_twins)

In [ ]:
# all 0/0 0/1 genotypes 
for k in range(1,12):
    # read twins 
    f_twins = f'{bucket}/data/twins_dups_samples_{k}.txt'
    twins_all = pandas.read_table(f_twins, sep = ',')
    twins_all['idv1'] = twins_all['IID1'].astype(str)
    twins_all['idv2'] = twins_all['IID2'].astype(str)
    twins_filtered = twins_all
    # read mt sub 
    mt_path =f'{bucket}/data/twins_dups_rep_samples_{k}.mt'
    mt = hl.read_matrix_table(mt_path)
    # now parse 
    for j in range(twins_filtered.shape[0]):
        idv1 = twins_filtered["idv1"].iloc[j]
        idv2 = twins_filtered["idv2"].iloc[j]
        print(idv1)
        print(idv2)
        twins = [str(idv1), str(idv2)]
        
        tsv_path = f'{bucket}/twins_dups/difs/{idv1}_{idv2}_difs.tsv'
        t_in = hl.import_table(tsv_path, impute=True)
        split_vid = t_in.locus.split(":")
        t_in = t_in.annotate(
        locus = hl.locus(split_vid[0], hl.int(split_vid[1]), reference_genome='GRCh38'))
        t_loci = t_in.select('locus').key_by('locus').distinct()
        
        # subset to samples 
        mt_subset_sib = mt.filter_cols(hl.literal(twins).contains(mt.s))  
        mt_subset_sib = mt_subset_sib.filter_rows(hl.is_defined(t_loci[mt_subset_sib.locus])) 

        # remove loci with missing genotypes
        # gq > 30 and defined 
        mt_subset_sib = mt_subset_sib.annotate_entries(GT = hl.or_missing(mt_subset_sib.GQ >= 30, mt_subset_sib.GT))
        mt_subset_sib = mt_subset_sib.filter_rows(hl.agg.sum(hl.is_defined(mt_subset_sib.GT)) == 2, keep = True)
        
        # these are the filters specific to these two, checking if different
        mt_subset_sib = mt_subset_sib.annotate_rows(is_non_ref = hl.agg.sum((mt_subset_sib.GT.is_non_ref())))    
        mt_subset_sib = mt_subset_sib.filter_rows(mt_subset_sib.is_non_ref == 1, keep = True)

        # subset to those that have a passing ft 
        mt_subset_sib = mt_subset_sib.filter_rows(hl.agg.sum(mt_subset_sib.FT == "FAIL") == 0, keep = True)

        # repartition to join faster 
        mt_subset_sib = mt_subset_sib.repartition(50)
        
        mt_subset_sib.GQ.export(f'{bucket}/twins_dups/difs/{idv1}_{idv2}_difs.tsv')

In [ ]:
# all alts for 0/0 0/1 genotypes 
for k in range(1,12):
    # read twins 
    f_twins = f'{bucket}/data/twins_dups_samples_{k}.txt'
    twins_all = pandas.read_table(f_twins, sep = ',')
    twins_all['idv1'] = twins_all['IID1'].astype(str)
    twins_all['idv2'] = twins_all['IID2'].astype(str)
    twins_filtered = twins_all
    # read mt sub 
    mt_path =f'{bucket}/data/twins_dups_rep_samples_{k}.mt'
    mt = hl.read_matrix_table(mt_path)
    # now parse 
    for j in range(twins_filtered.shape[0]):
        idv1 = twins_filtered["idv1"].iloc[j]
        idv2 = twins_filtered["idv2"].iloc[j]
        print(idv1)
        print(idv2)
        twins = [str(idv1), str(idv2)]
        
        tsv_path = f'{bucket}/twins_dups/difs/{idv1}_{idv2}_difs.tsv'
        t_in = hl.import_table(tsv_path, impute=True)
        split_vid = t_in.locus.split(":")
        t_in = t_in.annotate(
        locus = hl.locus(split_vid[0], hl.int(split_vid[1]), reference_genome='GRCh38'))
        t_loci = t_in.select('locus').key_by('locus').distinct()
        
        # subset to samples 
        mt_subset_sib = mt.filter_cols(hl.literal(twins).contains(mt.s))  
        mt_subset_sib = mt_subset_sib.filter_rows(hl.is_defined(t_loci[mt_subset_sib.locus])) 

        # remove loci with missing genotypes
        # gq > 30 and defined 
        mt_subset_sib = mt_subset_sib.annotate_entries(GT = hl.or_missing(mt_subset_sib.GQ >= 30, mt_subset_sib.GT))
        mt_subset_sib = mt_subset_sib.filter_rows(hl.agg.sum(hl.is_defined(mt_subset_sib.GT)) == 2, keep = True)
        
        # keep anything nonref 
        mt_subset_sib = mt_subset_sib.filter_rows(hl.agg.any(mt_subset_sib.GT.is_non_ref()))   

        # subset to those that have a passing ft 
        mt_subset_sib = mt_subset_sib.filter_rows(hl.agg.sum(mt_subset_sib.FT == "FAIL") == 0, keep = True)

        # repartition to join faster 
        mt_subset_sib = mt_subset_sib.repartition(50)
        
        mt_subset_sib.GT.export(f'{bucket}/twins_dups/difs_all_alt/{idv1}_{idv2}_difs_all_alt.tsv')

In [ ]:
%%bash 

mkdir -p twins_dups 

gsutil -m cp -r $WORKSPACE_BUCKET/twins_dups/difs ./twins_dups/
gsutil -m cp -r $WORKSPACE_BUCKET/twins_dups/difs_all_alt ./twins_dups/

mkdir -p ./twins_dups/formatted_difs